# Fine-tuned Model Inference Test

Sau khi mô hình chạy xong tiến trình huấn luyện ở file `02_finetuning_qlora.ipynb` và xuất ra thư mục **Adapter (LoRA weights)**, file Notebook này sẽ giúp bạn ghép nối nó vào Base Model gốc để xem kết quả chất lượng tóm tắt.

In [3]:
import sys
import os
import torch
from peft import PeftModel

# Add project root to sys.path for module imports
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from modules.model_loader import load_model_and_tokenizer
from modules.dataset_utils import format_prompt

### Step 1: Load Base Model and Tokenizer
Chúng ta cần tải bản gốc chuẩn bị trước. (Speed sẽ rất nhanh nếu `model_name` đã quét được từ trong cache/ổ cứng của bạn).

In [6]:
model_name = "Qwen/Qwen2.5-3B" # Thay bằng tên folder base model của bạn nếu cần tải nạp offline

base_model, tokenizer = load_model_and_tokenizer(
    model_name=model_name,
    use_4bit=True,
    torch_dtype="float16",
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:06<00:00, 62.00it/s]


### Step 2: Load Fine-tuned LoRA Adapter into the Base Model
Use `PeftModel` to layer the fine-tuned LoRA adapter on top of the base model.

In [7]:
lora_dir = os.path.join(PROJECT_ROOT, "models", "qwen25-3b-v1")

try:
    model = PeftModel.from_pretrained(base_model, lora_dir)
    print("Fine-tuned LoRA adapter loaded successfully!")
except Exception as e:
    print(f"Không tìm thấy thư mục LoRA tại: {lora_dir}. Quá trình train của bạn đã tạo ra file chưa?")
    raise e

Fine-tuned LoRA adapter loaded successfully!


### Step 3: Run a Meeting Summarization Test
Below is a sample meeting transcript to use as input for the model.

In [4]:
# You can replace this with an actual .txt meeting file of your own
test_meeting_text = """
# Cải thiện chất lượng phục vụ tại chuỗi quán cà phê

## I. Nội dung chính

### 1. Mục tiêu cuộc họp
- Giải quyết các khiếu nại của khách hàng về thái độ phục vụ và tốc độ lên món.

### 2. Các vấn đề đã thảo luận
- Khách hàng phản hồi tiêu cực về thái độ của nhân viên (sử dụng điện thoại, thiếu chào hỏi) tại quận 1 và quận 3.
- Speed lên món chậm (đợi 15-20 phút), nguyên nhân do quy trình quầy bar chưa tối ưu và nhân viên mới thiếu kỹ năng.

### 3. Kết luận và quyết định
- Xây dựng lại bộ quy chuẩn đào tạo nhân sự, bổ sung kỹ năng mềm và tình huống thực tế.
- Thiết kế lại quầy bar theo mô hình dây chuyền một chiều để tối ưu hóa thao tác.

## II. Danh sách công việc cần làm

| Công việc | Người phụ trách | Hạn chót |
| :--- | :--- | :--- |
| Soạn lại giáo trình đào tạo kỹ năng mềm cho nhân viên | Chị Lan (Phòng Nhân sự) | 25/05/2024 |
| Thiết kế lại sơ đồ quầy bar và báo giá thi công | Chị Mai (Phòng Vận hành) | 28/05/2024 |

[08:20] Ý tưởng hay đó Mai, vầy nè, em làm việc với bên thiết kế xem có phương án nào tối ưu không, rồi báo giá cho anh luôn nha.
[08:22] Dạ, để em triển khai cái này liền, chắc tầm cuối tuần là có bản vẽ sơ bộ cho anh coi đó.
[08:24] Còn một cái nữa, anh thấy khách họ cũng nhắc nhiều về cái vụ vệ sinh trong quán, nhất là cái nhà vệ sinh á, nhiều khi vô thấy dơ mà không có ai dọn dẹp thường xuyên.
[08:26] Ờ đúng rồi, cái này quan trọng nè. Em nghĩ mình nên có cái checklist dọn dẹp mỗi 30 phút một lần, rồi dán ở sau cửa luôn, bạn nào dọn xong thì ký tên vô đó để mình kiểm soát.
[08:28] Đúng rồi đó Lan, làm vậy đi cho nó chuyên nghiệp. À mà mình cũng nên có cái chương trình thưởng cho "Nhân viên xuất sắc của tháng" dựa trên đánh giá của khách hàng nữa.
[08:30] Dạ, em định là mình sẽ để một cái mã QR ở mỗi bàn á anh, khách họ quét mã đó để đánh giá phục vụ luôn, nếu bạn nào được khen nhiều thì mình thưởng nóng luôn cho tụi nhỏ nó có động lực.
[08:32] Hay đó, vầy mới đúng là cái anh cần nè. Công nghệ vô chút cho nó hiện đại. Team IT bên mình có làm cái này được không ta?
[08:34] Dạ được anh, cái này đơn giản mà, để em báo bên đó làm cái form khảo sát rồi tạo mã QR cho từng chi nhánh luôn.
[08:36] Ok, vậy chốt lại mấy cái đó nha. Mà Lan nè, em nhớ nhắc mấy bạn nhân viên là tuyệt đối không được dùng điện thoại trong giờ làm việc trừ trường hợp khẩn cấp nha, anh thấy cái này là cái gây khó chịu nhất cho khách luôn á.
[08:38] Dạ em biết rồi anh, em sẽ ra cái quy định mới và áp dụng hình thức kỷ luật nghiêm nếu bạn nào vi phạm cái này.
"""

# Pass an empty string for output_text since we want the model to generate it
prompt = format_prompt(input_text=test_meeting_text, output_text="")

# Display the raw prompt that will be fed to the model
print(prompt)

Hãy cập nhật lại báo cáo cuộc họp trước đó bằng cách bổ sung thêm thông tin từ nội dung thảo luận mới dưới đây.

### Đầu vào (Báo cáo cũ & Nội dung thảo luận mới):
# Cải thiện chất lượng phục vụ tại chuỗi quán cà phê

## I. Nội dung chính

### 1. Mục tiêu cuộc họp
- Giải quyết các khiếu nại của khách hàng về thái độ phục vụ và tốc độ lên món.

### 2. Các vấn đề đã thảo luận
- Khách hàng phản hồi tiêu cực về thái độ của nhân viên (sử dụng điện thoại, thiếu chào hỏi) tại quận 1 và quận 3.
- Tốc độ lên món chậm (đợi 15-20 phút), nguyên nhân do quy trình quầy bar chưa tối ưu và nhân viên mới thiếu kỹ năng.

### 3. Kết luận và quyết định
- Xây dựng lại bộ quy chuẩn đào tạo nhân sự, bổ sung kỹ năng mềm và tình huống thực tế.
- Thiết kế lại quầy bar theo mô hình dây chuyền một chiều để tối ưu hóa thao tác.

## II. Danh sách công việc cần làm

| Công việc | Người phụ trách | Hạn chót |
| :--- | :--- | :--- |
| Soạn lại giáo trình đào tạo kỹ năng mềm cho nhân viên | Chị Lan (Phòng Nhân sự) | 25

### Step 4: Run Inference (Generate the Summary)

In [6]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

input_length = inputs["input_ids"].shape[1]

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,        # Maximum tokens for the generated summary
        temperature=0.3,           # Low temperature for stable, focused output
        top_p=0.9,                 # Filter unlikely tokens
        repetition_penalty=1.1,    # Penalise repetition
        pad_token_id=tokenizer.pad_token_id,
    )

# Strip the prompt prefix — keep only the newly generated text
generated_tokens = outputs[0, input_length:]
result = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("====== AI-GENERATED SUMMARY ======")
print(result)

====== BẢN TÓM TẮT AI TẠO RA ======
# Cải thiện chất lượng phục vụ tại chuỗi quán cà phê

## I. Nội dung chính

### 1. Mục tiêu cuộc họp
- Giải quyết các khiếu nại của khách hàng về thái độ phục vụ và tốc độ lên món.
- Thống nhất các biện pháp đào tạo nhân sự và cải tiến quy trình vận hành.

### 2. Các vấn đề đã thảo luận
- Khách hàng phản hồi tiêu cực về thái độ của nhân viên (sử dụng điện thoại, thiếu chào hỏi) tại quận 1 và quận 3.
- Tốc độ lên món chậm (đợi 15-20 phút), nguyên nhân do quy trình quầy bar chưa tối ưu và nhân viên mới thiếu kỹ năng.
- Tình trạng vệ sinh không đồng đều giữa các chi nhánh (không có lịch dọn dẹp cố định).
- Sự cần thiết của việc sử dụng công nghệ để quản lý và nhận xét khách hàng.

### 3. Kết luận và quyết định
- Xây dựng lại bộ quy chuẩn đào tạo nhân sự, bổ sung kỹ năng mềm và tình huống thực tế.
- Thiết kế lại quầy bar theo mô hình dây chuyền một chiều để tối ưu hóa thao tác.
- Triển khai hệ thống đánh giá khách hàng qua mã QR tích hợp công nghệ.
- Áp 

### Step 5: Evaluate on Test Set (ROUGE + BERTScore)

Các cell dưới đây sẽ:
- Đọc test set từ thư mục dữ liệu gốc.
- Sinh summary bằng model fine-tuned.
- Tính ROUGE và BERTScore cho tiếng Việt.

In [4]:
# Nếu thiếu thư viện, bỏ comment để cài
# %pip install -q evaluate rouge_score bert-score

import re
import glob
import numpy as np
import evaluate
from tqdm.auto import tqdm

from modules.dataset_utils import parse_meeting_file, format_prompt


def clean_generated_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    return text


def generate_summary(model, tokenizer, input_text: str, max_new_tokens: int = 512) -> str:
    prompt = format_prompt(input_text=input_text, output_text="")
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
        )

    gen_tokens = outputs[0, input_length:]
    pred = tokenizer.decode(gen_tokens, skip_special_tokens=True)
    return clean_generated_text(pred)


def load_test_samples(data_dir: str, limit: int | None = 30):
    files = sorted(glob.glob(os.path.join(data_dir, "*.txt")))
    samples = []

    for fp in files:
        src, ref = parse_meeting_file(fp)
        if src and ref:
            samples.append({"file": fp, "input": src, "reference": ref.strip()})

    if limit is not None:
        samples = samples[:limit]

    return samples

In [8]:
# Chọn đúng thư mục test data của bạn
TEST_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "v3")
MAX_TEST_SAMPLES = 20  # tăng nếu muốn đánh giá đầy đủ hơn

samples = load_test_samples(TEST_DATA_DIR, limit=MAX_TEST_SAMPLES)
print(f"Loaded {len(samples)} test samples from: {TEST_DATA_DIR}")

predictions = []
references = []

for item in tqdm(samples, desc="Generating summaries"):
    pred = generate_summary(model, tokenizer, item["input"], max_new_tokens=512)
    predictions.append(pred)
    references.append(item["reference"])

print("Done generating predictions.")
print("Example prediction:\n", predictions[0][:500] if predictions else "N/A")

Loaded 20 test samples from: c:\Users\ezycloudx-admin\Downloads\qwen2.5-3b-meeting-summarization\data\raw\v3


Generating summaries: 100%|██████████| 20/20 [13:45<00:00, 41.26s/it]

Done generating predictions.
Example prediction:
 # Chiến lược ra mắt sản phẩm gia dụng thông minh và chiến thuật cạnh tranh ## I. Nội dung chính ### 1. Mục tiêu cuộc họp - Định hướng chiến lược bán hàng cho dòng sản phẩm gia dụng thông minh mới. - Thảo luận phương án ứng phó với áp lực cạnh tranh từ các thương hiệu khác về giá. ### 2. Các vấn đề đã thảo luận - Thị trường: Sức mua hồi phục, khách hàng ưu tiên tính năng tiết kiệm điện và thiết kế. - Đối thủ: Thương hiệu X đang cạnh tranh bằng các chương trình giảm giá sâu (20%). ### 3. Kết luận 


In [9]:
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

rouge_result = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=False,
)

bertscore_result = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="vi",
    model_type="xlm-roberta-large",
)

metrics = {
    "rouge1": rouge_result.get("rouge1"),
    "rouge2": rouge_result.get("rouge2"),
    "rougeL": rouge_result.get("rougeL"),
    "bertscore_precision": float(np.mean(bertscore_result["precision"])) if bertscore_result.get("precision") else None,
    "bertscore_recall": float(np.mean(bertscore_result["recall"])) if bertscore_result.get("recall") else None,
    "bertscore_f1": float(np.mean(bertscore_result["f1"])) if bertscore_result.get("f1") else None,
    "num_samples": len(predictions),
}

print("===== TEST SET METRICS =====")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

c:\Users\ezycloudx-admin\Downloads\qwen2.5-3b-meeting-summarization\venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ezycloudx-admin\.cache\huggingface\hub\models--xlm-roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4159.00it/s]


===== TEST SET METRICS =====
rouge1: 0.8413
rouge2: 0.6758
rougeL: 0.6671
bertscore_precision: 0.9403
bertscore_recall: 0.9524
bertscore_f1: 0.9463
num_samples: 20
